# Create a dataset for EVASPA with landsat8 data 
* On a specific S2 tile ID
* On S2 tiles corresponding to a specific ROI

### Imports

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
import sys

sys.path.append("../src")

In [ ]:
import os
import geopandas as gpd
from tqdm import tqdm
from sensorsio import mgrs
import logging

In [ ]:
from etdataset.api import create_dataset
from etdataset.plot import plot_images
from etdataset.writer import write_dataset, export_matlab
from etdataset.utils import get_bbox_from_roi

In [ ]:
if os.environ["ETDATASET_TEST_DATA_PATH"] is None:
    raise Exception("Variable ETDATASET_TEST_DATA_PATH must be set")
etdataset_data_path = os.environ["ETDATASET_TEST_DATA_PATH"]

### Inputs 
* Landsat product path
* S2 tile id or ROI path

In [ ]:
# Path to Landsat product
ls8_path = os.path.join(
    etdataset_data_path, "Landsat", "LC09_L2SP_205050_20230327_20230329_02_T1"
)

In [ ]:
# Path to ERA5 product
era5_path = os.path.join(
    etdataset_data_path,
    "era5",
    "download_era5_2023-03-27_14.57_-16.96_13.37_-15.73.nc",
)

In [ ]:
# S2 tile Id or ROI shapefile
s2_tile_id = "28PCA"
roi_path = os.path.join(etdataset_data_path, "zones", "Zone_Senegal_Centre.shp")

In [ ]:
# Output directory for results
output_dir = "out"
os.makedirs(output_dir, exist_ok=True)

### Create dataset from Landsat8 product for a specific S2 tile ID

In [ ]:
# To take or not into account mask, use option use_mask
ls8 = create_dataset(vis_path=ls8_path, tile_id=s2_tile_id)

In [ ]:
ls8

In [ ]:
plot_images(ls8)

In [ ]:
# To take or not into account mask, use option use_mask
ls8 = create_dataset(
    vis_path=ls8_path, tile_id=s2_tile_id, radiation=era5_path
)

In [ ]:
ls8

In [ ]:
plot_images(ls8)

In [ ]:
# Tif format
write_dataset(ls8, directory=output_dir, separate=True)
write_dataset(ls8, directory=output_dir, separate=False)

In [ ]:
# Matlab
export_matlab(ls8, directory=output_dir)

### Create datasets from Landsat8 product for ROI

The Landsat product is tiled on MRGS tiles. Only tiles with more than 20% overlap are processed.

In [ ]:
roi_bbox, roi_crs = get_bbox_from_roi(roi_path)

In [ ]:
tile_ids = mgrs.get_mgrs_tiles_from_roi(roi_bbox, roi_crs)
tile_ids

Keep tiles ID with more than 20% of overlap

In [ ]:
tile_ids = tile_ids[tile_ids["overlap_percentage"] >= 20.0]

In [ ]:
tile_ids

Create the datasets 

In [ ]:
datasets = []
for tile_id in tqdm(tile_ids.Name.values):
    ls8 = create_dataset(vis_path=ls8_path, tile_id=tile_id, radiation=era5_path)
    datasets.append(ls8)

Write results

In [ ]:
for ds in tqdm(datasets):
    write_dataset(ds, directory=output_dir)